# Yeast-PCM-TEG Simulation

Model of yeast fermentation coupled with a phase-change material (PCM) buffer and thermoelectric generator (TEG).

In [ ]:
%matplotlib inline
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

In [ ]:
V_broth = 0.50
m_broth = 0.50
C_broth = m_broth * 4184.0

S_initial = 100.0
X_initial = 4.0
X_max = 22.0
delta_H_glyc = 655.0

m_pcm = 0.06
T_pcm_melt = 28.0
delta_T_melt = 1.0
L_f = 180000.0
Cp_pcm_sensible = 2000.0

R_insul = 35.0
R_b_pcm = 2.0
R_th_teg = 2.2
R_contact = 0.5
R_sink_air = 2.0
C_sink = 0.15 * 900.0

S_teg = 0.045
R_int = 2.5
T_amb = 22.0
T_pitch = 26.0

In [ ]:
def cardinal_growth_rate(T):
    if T <= 10.0 or T >= 38.5:
        return 0.0
    num = (T - 38.5) * (T - 10.0)**2
    den = (32.0 - 10.0) * ((32.0 - 10.0)*(T - 32.0) - (32.0 - 38.5)*(32.0 + 10.0 - 2.0*T))
    if den == 0.0:
        return 0.0
    return 0.48 * max(0.0, num / den)

def odes_buf(t, y):
    X, S, Tb, Tpcm, Tc = y
    if S > 0.001 and X < X_max:
        mu = cardinal_growth_rate(Tb) * (S / (2.0 + S))
        dX_dt = (mu * X * (1.0 - X/X_max)) / 3600.0
        dS_dt = (-(1.0 / 0.16) * (dX_dt * 3600.0) - 0.015 * X) / 3600.0
    else:
        dX_dt, dS_dt = 0.0, 0.0
    Q_bio = V_broth * delta_H_glyc * (-dS_dt)
    Q_loss = (Tb - T_amb) / R_insul
    Q_to_pcm = (Tb - Tpcm) / R_b_pcm
    Q_teg = (Tpcm - Tc) / (R_th_teg + R_contact)
    Q_sink = (Tc - T_amb) / R_sink_air
    
    sigma = delta_T_melt / 2.0
    dbeta_dT = (1.0 / (sigma * np.sqrt(2.0 * np.pi))) * np.exp(-0.5 * ((Tpcm - T_pcm_melt)/sigma)**2)
    C_pcm_dyn = m_pcm * (Cp_pcm_sensible + L_f * dbeta_dT)
    
    dTb = (Q_bio - Q_loss - Q_to_pcm) / C_broth
    dTpcm = (Q_to_pcm - Q_teg) / C_pcm_dyn
    dTc = (Q_teg - Q_sink) / C_sink
    return [dX_dt, dS_dt, dTb, dTpcm, dTc]

def odes_unbuf(t, y):
    X, S, Tb, Tc = y
    if S > 0.001 and X < X_max:
        mu = cardinal_growth_rate(Tb) * (S / (2.0 + S))
        dX_dt = (mu * X * (1.0 - X/X_max)) / 3600.0
        dS_dt = (-(1.0 / 0.16) * (dX_dt * 3600.0) - 0.015 * X) / 3600.0
    else:
        dX_dt, dS_dt = 0.0, 0.0
    Q_bio = V_broth * delta_H_glyc * (-dS_dt)
    Q_loss = (Tb - T_amb) / R_insul
    Q_teg = (Tb - Tc) / (R_b_pcm + R_th_teg + R_contact)
    Q_sink = (Tc - T_amb) / R_sink_air
    dTb = (Q_bio - Q_loss - Q_teg) / C_broth
    dTc = (Q_teg - Q_sink) / C_sink
    return [dX_dt, dS_dt, dTb, dTc]

In [ ]:
t_eval = np.linspace(0.0, 20.0 * 3600.0, 1000)
sol_unbuf = solve_ivp(odes_unbuf, (0.0, 20.0 * 3600.0), [X_initial, S_initial, T_pitch, T_amb], t_eval=t_eval, method='Radau')
sol_buf = solve_ivp(odes_buf, (0.0, 20.0 * 3600.0), [X_initial, S_initial, T_pitch, T_amb, T_amb], t_eval=t_eval, method='Radau')

t_hours = t_eval / 3600.0
deltaT_unbuf = np.maximum(0.0, sol_unbuf.y[2] - sol_unbuf.y[3])
deltaT_buf = np.maximum(0.0, sol_buf.y[3] - sol_buf.y[4])

P_unbuf = ((S_teg * deltaT_unbuf)**2) / (4.0 * R_int)
P_buf = ((S_teg * deltaT_buf)**2) / (4.0 * R_int)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes[0, 0].plot(t_hours, sol_unbuf.y[2], 'r--', label='Broth (Unbuffered)')
axes[0, 0].plot(t_hours, sol_buf.y[2], 'b-', label='Broth (PCM-Buffered)')
axes[0, 0].plot(t_hours, sol_buf.y[3], 'g-.', label='PCM Buffer')
axes[0, 0].axhline(T_pcm_melt, color='gray', linestyle=':', label='Melt (28°C)')
axes[0, 0].set_title("Thermal Pulse Profiles [°C]")
axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(t_hours, deltaT_unbuf, 'r--', label='ΔT Unbuffered')
axes[0, 1].plot(t_hours, deltaT_buf, 'b-', label='ΔT Buffered')
axes[0, 1].set_title("Thermal Gradient Across TEG [°C]")
axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(t_hours, P_unbuf * 1000, 'r--', label='Unbuffered')
axes[1, 0].plot(t_hours, P_buf * 1000, 'b-', label='Buffered')
axes[1, 0].set_title("Instantaneous Power [mW]")
axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(t_hours, np.cumsum(P_unbuf * np.gradient(t_eval)), 'r--', label='Unbuffered')
axes[1, 1].plot(t_hours, np.cumsum(P_buf * np.gradient(t_eval)), 'b-', label='Buffered')
axes[1, 1].set_title("Cumulative Harvested Energy [J]")
axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()